**Sample ID**: 1046




**Query**: In Google Drive, identify all design files in the folder "Campaign Assets – To Review" whose size is greater than 5 MB. Move each of these files to the folder "Campaign Assets – Heavy", and for every moved file, append its Name and File Size (in MB) to the Google Sheet "Large Assets Log".




**DB Type**: Base Case




**Case Description**: There is at least one file in the Google Drive folder "Campaign Assets – To Review" whose name contains the word "design" (case-insensitive) and whose file size is greater than 5 MB. No other files are considered relevant. Only one Google Drive folder titled "Campaign Assets – Heavy" exists. Only one Google Sheet titled "Large Assets Log" exists, and it contains only one tab. The relevant columns are "File Name" and "File Size (MB)", with range = 'Sheet1!A1:B1'. No other columns are relevant. Each matching file is to be moved to the "Campaign Assets – Heavy" folder, and its Name and File Size (rounded to MB) are to be appended to the "Large Assets Log" sheet.




**Global/Context Variables**:


- folder_to_review_name = "Campaign Assets – To Review"
- folder_heavy_name = "Campaign Assets – Heavy"
- sheet_name = "Large Assets Log"
- minimum_size = 5



**APIs**:

- gdrive
- google_sheets


# Set Up

## Download relevant files

In [1]:
import io
import os
import sys
import zipfile
import shutil
import re
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Version to download
VERSION = "0.1.4"  # Version of the API

# Define paths
CONTENT_DIR = '/content'
APIS_DIR = os.path.join(CONTENT_DIR, 'APIs')
DBS_DIR = os.path.join(CONTENT_DIR, 'DBs')
SCRIPTS_DIR = os.path.join(CONTENT_DIR, 'Scripts')
FC_DIR = os.path.join(CONTENT_DIR, 'Schemas')
ZIP_PATH = os.path.join(CONTENT_DIR, f'APIs_V{VERSION}.zip')

# Google Drive Folder ID where versioned APIs zip files are stored
APIS_FOLDER_ID = '1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4'

# List of items to extract from the zip file
ITEMS_TO_EXTRACT = ['APIs/', 'DBs/', 'Scripts/', 'Schemas/']

# Clean up existing directories and files
for path in [APIS_DIR, DBS_DIR, SCRIPTS_DIR, FC_DIR, ZIP_PATH]:
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

# Authenticate and create the drive service
auth.authenticate_user()
drive_service = build('drive', 'v3')

# Helper function to download a file from Google Drive
def download_drive_file(service, file_id, output_path, file_name=None, show_progress=True):
    """Downloads a file from Google Drive"""
    destination = output_path
    request = service.files().get_media(fileId=file_id)
    with io.FileIO(destination, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if show_progress:
                print(f"Download progress: {int(status.progress() * 100)}%")


# 1. List files in the specified APIs folder
print(f"Searching for APIs zip file with version {VERSION} in folder: {APIS_FOLDER_ID}...")
apis_file_id = None

try:
    query = f"'{APIS_FOLDER_ID}' in parents and trashed=false"
    results = drive_service.files().list(q=query, fields="files(id, name)").execute()
    files = results.get('files', [])
    for file in files:
        file_name = file.get('name', '')
        if file_name.lower() == f'apis_v{VERSION.lower()}.zip':
            apis_file_id = file.get('id')
            print(f"Found matching file: {file_name} (ID: {apis_file_id})")
            break

except Exception as e:
    print(f"An error occurred while listing files in Google Drive: {e}")

if not apis_file_id:
    print(f"Error: Could not find APIs zip file with version {VERSION} in the specified folder.")
    sys.exit("Required APIs zip file not found.")

# 2. Download the found APIs zip file
print(f"Downloading APIs zip file with ID: {apis_file_id}...")
download_drive_file(drive_service, apis_file_id, ZIP_PATH, file_name=f'APIs_V{VERSION}.zip')

# 3. Extract specific items from the zip file to /content
print(f"Extracting specific items from {ZIP_PATH} to {CONTENT_DIR}...")
try:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()

        for member in zip_contents:
            extracted = False
            for item_prefix in ITEMS_TO_EXTRACT:
              if member == item_prefix or member.startswith(item_prefix):
                    zip_ref.extract(member, CONTENT_DIR)
                    extracted = True
                    break

except zipfile.BadZipFile:
    print(f"Error: The downloaded file at {ZIP_PATH} is not a valid zip file.")
    sys.exit("Invalid zip file downloaded.")
except Exception as e:
    print(f"An error occurred during extraction: {e}")
    sys.exit("Extraction failed.")


# 4. Clean up
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

# 5. Add APIs to path
if os.path.exists(APIS_DIR):
    sys.path.append(APIS_DIR)
else:
    print(f"Error: APIS directory not found at {APIS_DIR} after extraction. Cannot add to path.")

# 6. Quick verification
# Check for the presence of the extracted items
verification_paths = [APIS_DIR, DBS_DIR, SCRIPTS_DIR]
all_present = True
print("\nVerifying extracted items:")
for path in verification_paths:
    if os.path.exists(path):
        print(f"✅ {path} is present.")
    else:
        print(f"❌ {path} is MISSING!")
        all_present = False

if all_present:
    print(f"\n✅ Setup complete! Required items extracted to {CONTENT_DIR}.")
else:
    print("\n❌ Setup failed! Not all required items were extracted.")

# 7. Generate Schemas

print("\nGenerating FC Schemas")

# Change working directory to the source folder

# Iterate through the packages in the /content/APIs directory

    # Check if it's a directory (to avoid processing files)
        # Call the function to generate schema for the current package
print(f"✅ Successfully generated {len(os.listdir(FC_DIR))} FC Schemas to {FC_DIR}")
os.chdir(CONTENT_DIR)

Searching for APIs zip file with version 0.1.4 in folder: 1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4...
Found matching file: APIs_V0.1.4.zip (ID: 1TnAaWGfVrMxWTilyhy46-Aue_bh0XkNk)
Download progress: 100%
Extracting specific items from /content/APIs_V0.1.4.zip to /content...

Verifying extracted items:
✅ /content/APIs is present.
✅ /content/DBs is present.
✅ /content/Scripts is present.

✅ Setup complete! Required items extracted to /content.

Generating FC Schemas
✅ Successfully generated 70 FC Schemas to /content/Schemas


## Install Dependencies and Clone Repositories

In [2]:
!pip install -r /content/APIs/requirements.txt

## Import APIs and initiate DBs

In [3]:
import gdrive
import google_sheets

# Load the simulation data for Google Drive and Google Sheets
gdrive.SimulationEngine.db.load_state("/content/DBs/GDriveDefaultDB.json")
google_sheets.SimulationEngine.db.load_state("/content/DBs/SheetsDefaultDB.json")

# Global variables
folder_to_review_name = "Campaign Assets – To Review"
folder_heavy_name = "Campaign Assets – Heavy"
sheet_name = "Large Assets Log"

# Local Variable
columns_needed = ['File Name', 'File Size (MB)']

# Step 1: Ensure "Campaign Assets – To Review" folder exists
query_to_review = f"name='{folder_to_review_name}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
folders_response = gdrive.list_user_files(q=query_to_review)
folders = folders_response.get("files", []) if isinstance(folders_response, dict) else []

# Check if folder exists or create it
to_review_folder = next((folder for folder in folders if folder["name"] == folder_to_review_name), None)
if not to_review_folder:
    to_review_folder = gdrive.create_file_or_folder({
        "name": folder_to_review_name,
        "mimeType": "application/vnd.google-apps.folder"
    })
    print(f"Created folder: {folder_to_review_name}")
else:
    print(f"Found existing folder: {folder_to_review_name}")

print(to_review_folder)
# Simulate uploading a file to the folder for testing purposes
file_data = {
    "name": "design_mockup_6MB.png",
    "size": str(6 * 1024 * 1024),  # 6MB file
    "parents": [to_review_folder["id"]]  # Ensure the file is added to the correct folder
}
file_creation_response = gdrive.create_file_or_folder(file_data)
print("File created:", file_creation_response)

# Step 3: Ensure "Campaign Assets – Heavy" folder exists or create it if not
folder_to_heavy_name = "Campaign Assets – Heavy"
folders_response_heavy = gdrive.list_user_files(q=f"name='{folder_to_heavy_name}' and trashed=false")
folders_heavy = folders_response_heavy.get("files", []) if isinstance(folders_response_heavy, dict) else []

to_heavy_folder = next((folder for folder in folders_heavy if folder["name"] == folder_to_heavy_name), None)
if not to_heavy_folder:
    to_heavy_folder = gdrive.create_file_or_folder({
        "name": folder_to_heavy_name,
        "mimeType": "application/vnd.google-apps.folder"
    })
    print(f"Created folder: {folder_to_heavy_name}")

# Step 4: Ensure "Large Assets Log" Google Sheet exists or create it if not
sheets_response = gdrive.list_user_files(q=f"name='{sheet_name}' and trashed=false")
sheets = sheets_response.get("files", []) if isinstance(sheets_response, dict) else []

log_sheet = next((sheet for sheet in sheets if sheet["name"] == sheet_name), None)
if not log_sheet:
    log_sheet = gdrive.create_file_or_folder({
        "name": sheet_name,
        "mimeType": "application/vnd.google-apps.spreadsheet"
    })
    print(f"Created Google Sheet: {sheet_name}")

# Update sheet with columns
spreadsheet_id = log_sheet["id"]
sheet_range = "Sheet1!A1:B1"  # Assuming the sheet's name is "Sheet1" and the columns go into A1:B1

# Correct update call: Send the values directly in a list
google_sheets.update_spreadsheet_values(
    spreadsheet_id=spreadsheet_id,
    range=sheet_range,
    values=[columns_needed],  # Directly passing the values
    valueInputOption="RAW"  # Specify how the values should be input
)
print(f"Inserted columns: {columns_needed}")

Created folder: Campaign Assets – To Review
{'kind': 'drive#file', 'id': 'file_3', 'driveId': '', 'name': 'Campaign Assets – To Review', 'mimeType': 'application/vnd.google-apps.folder', 'parents': [], 'createdTime': '2025-03-14T00:00:00Z', 'modifiedTime': '2025-03-14T00:00:00Z', 'trashed': False, 'starred': False, 'owners': ['john.doe@gmail.com'], 'size': '0', 'md5Checksum': '', 'sha1Checksum': '', 'sha256Checksum': '', 'imageMediaMetadata': {}, 'videoMediaMetadata': {}, 'permissions': [{'id': 'permission_file_3', 'role': 'owner', 'type': 'user', 'emailAddress': 'john.doe@gmail.com'}], 'enforceSingleParent': False, 'ignoreDefaultVisibility': False, 'keepRevisionForever': False, 'ocrLanguage': '', 'supportsAllDrives': False, 'supportsTeamDrives': False, 'useContentAsIndexableText': False, 'includePermissionsForView': '', 'includeLabels': '', 'revisionSettings': {'keepForever': False}, 'ocrMetadata': {}, 'indexableText': '', 'additionalPermissions': [], 'labels': []}
File created: {'kin

# Initial Assertion
1. Assert that a Google Drive folder titled "Campaign Assets – To Review" exists.  
2. Assert that at least one file in above folder has "design" in its name (case-insensitive) and a file size greater than 5 MB.  
3. Assert that a Google Drive folder titled "Campaign Assets – Heavy" exists.  
4. Assert that a Google Sheet titled "Large Assets Log" exists.  
5. Assert that the sheet contains only one tab and contains columns "File Name" and "File Size (MB)".  
6. Assert that the range of the sheet is "Sheet1!A1:B1"

In [4]:
from Scripts.assertions_utils import *
import gdrive
import google_sheets

# Global variables
folder_to_review_name = "Campaign Assets – To Review"
folder_to_heavy_name = "Campaign Assets – Heavy"
sheet_name = "Large Assets Log"
min_file_size_mb = 5
expected_range='Sheet1!A1:B1'

# Local variables
file_name_contains = "design"
columns_needed = ["File Name", "File Size (MB)"]

# Step 1: Assert that "Campaign Assets – To Review" folder exists
query_to_review = f"name='{folder_to_review_name}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
folders_response = gdrive.list_user_files(q=query_to_review)
folders = folders_response.get("files", []) if isinstance(folders_response, dict) else []

to_review_folder = next((folder for folder in folders if compare_strings(folder["name"], folder_to_review_name)), None)
assert to_review_folder, f"Folder '{folder_to_review_name}' not found in Google Drive."

# Step 2: Assert that at least one file in "Campaign Assets – To Review" has "design" in its name and size > 5MB
query_files_in_to_review = f"'{to_review_folder['id']}' in parents and name contains '{file_name_contains}' and trashed=false"
files_in_to_review_response = gdrive.list_user_files(q=query_files_in_to_review)
files_in_to_review = files_in_to_review_response.get("files", []) if isinstance(files_in_to_review_response, dict) else []

file_with_design = None
for file in files_in_to_review:
    if compare_is_string_subset("design", file.get("name", "")):
        file_size_str = file.get("size", "0")  # Default to "0" if no size is provided
        file_size_mb = int(file_size_str) / (1024 * 1024)  # Convert bytes to MB

        if file_size_mb > min_file_size_mb:
            file_with_design = file
            break

assert file_with_design, f"No file with 'design' in its name and size greater than {min_file_size_mb}MB found in folder '{folder_to_review_name}'."

# Step 3: Assert that "Campaign Assets – Heavy" folder exists
folders_response_heavy = gdrive.list_user_files(q=f"name='{folder_to_heavy_name}' and trashed=false")
folders_heavy = folders_response_heavy.get("files", []) if isinstance(folders_response_heavy, dict) else []

to_heavy_folder = next((folder for folder in folders_heavy if compare_strings(folder["name"], folder_to_heavy_name)), None)
assert to_heavy_folder, f"Folder '{folder_to_heavy_name}' not found or created in Google Drive."

# Step 4: Assert that the Google Sheet titled "Large Assets Log" exists
query_large_assets_log = f"name='{sheet_name}' and mimeType='application/vnd.google-apps.spreadsheet' and trashed=false"
sheets_response = gdrive.list_user_files(q=query_large_assets_log)
sheets = sheets_response.get("files", []) if isinstance(sheets_response, dict) else []

log_sheet = next((sheet for sheet in sheets if compare_strings(sheet["name"], sheet_name)), None)
assert log_sheet, f"Google Sheet '{sheet_name}' not found in Google Drive."

# Step 5: Assert that the sheet contains only one tab and columns "File Name" and "File Size (MB)"
spreadsheet_id = log_sheet["id"]
# Get spreadsheet details
spreadsheet_details = google_sheets.get_spreadsheet(spreadsheet_id=spreadsheet_id, includeGridData=True)
sheets_in_spreadsheet = spreadsheet_details.get("sheets", []) if isinstance(spreadsheet_details, dict) else []

# Extract sheet title
sheet_title = sheets_in_spreadsheet[0].get("properties", {}).get("title")

# Extract range key from 'data'
data_dict = spreadsheet_details.get("data", {})
sheet_range = next(iter(data_dict.keys()), None)
# Fetch header row using the extracted range
values_response = google_sheets.get_spreadsheet_values(spreadsheet_id=spreadsheet_id, range=sheet_range)
values = values_response.get("values", []) if isinstance(values_response, dict) else []
header_row = values[0] if values else []

# Validate required columns
assert len(sheets_in_spreadsheet) == 1 and compare_is_list_subset(columns_needed, header_row, list_comparison_function="all"), \
    f"Columns {columns_needed} not found in sheet '{sheet_name}'. Found: {header_row}"

# Assert that the range of the sheet is 'Sheet1!A1:B1'
assert compare_strings(sheet_range,expected_range)


# Action
- Identify all design files in the folder "Campaign Assets – To Review" whose size is greater than 5 MB.
- Move each of these files to the folder "Campaign Assets – Heavy", and for every moved file, append its Name and File Size (in MB) to the Google Sheet "Large Assets Log".

In [5]:
import gdrive
import google_sheets
import math
import json # Used for pretty printing

# Constants
source_folder_name = "Campaign Assets – To Review"
sheet_name = "Large Assets Log"

# --- Find Source Folder ID ---
source_folder_query = f"name='{source_folder_name}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
source_folder_response = gdrive.list_user_files(q=source_folder_query)
source_folder_id = source_folder_response['files'][0]['id']

file_query = f"'{source_folder_id}' in parents and mimeType!='application/vnd.google-apps.folder' and trashed=false"
# Request fields needed: id, name, size, parents
files_response = gdrive.list_user_files(q=file_query)

for file in files_response.get('files', []):
    file_id = file.get('id')
    name = file.get('name')
    size_bytes = int(file.get('size'))
    parents = file.get('parents')
    data = file.get('data')

    print(f"File ID: {file_id}")
    print(f"Name: {name}")
    print(f"Size Bytes: {size_bytes}")
    print(f"Parents: {parents}")
    print(f"Data: {data}")
    print("-" * 40)

# Retrieve the Large assets log sheet details
query_large_assets_log = f"name='{sheet_name}' and mimeType='application/vnd.google-apps.spreadsheet' and trashed=false"
sheets_response = gdrive.list_user_files(q=query_large_assets_log)
sheets = sheets_response.get("files", []) if isinstance(sheets_response, dict) else []
log_sheet = next((sheet for sheet in sheets if sheet["name"] == sheet_name), None)
spreadsheet_id = log_sheet["id"]
spreadsheet_details = google_sheets.get_spreadsheet(spreadsheet_id=spreadsheet_id, includeGridData=True)
print(spreadsheet_details)

File ID: file_4
Name: design_mockup_6MB.png
Size Bytes: 6291456
Parents: ['file_3']
Data: None
----------------------------------------
{'id': 'file_6', 'properties': None, 'sheets': [{'properties': {'sheetId': 'sheet1', 'title': 'Sheet1', 'index': 0, 'sheetType': 'GRID', 'gridProperties': {'rowCount': 1000, 'columnCount': 26}}}], 'data': {'Sheet1!A1:B1': [['File Name', 'File Size (MB)']]}}


In [6]:
# Model inferred

files_to_move=[{'id': 'file_4',
  'name': 'design_mockup_6MB.png',
  'size_bytes': 6291456,
  'current_parent': 'file_3'}]

target_range = 'Sheet1!A1:B1'

# Global Variable
destination_folder_name = "Campaign Assets – Heavy"
log_sheet_name = "Large Assets Log"
min_file_size_mb = 5

# Local
min_file_size_bytes = min_file_size_mb * 1024 * 1024


# --- Print Sampled Data ---
print(f"\nFound {len(files_to_move)} files to move and log:")
if files_to_move:
    # Print info for the first few files as a sample
    for i, file_info in enumerate(files_to_move[:3]):
        print(f"  - File {i+1}:")
        print(f"    ID: {file_info['id']}")
        print(f"    Name: {file_info['name']}")
        print(f"    Size (Bytes): {file_info['size_bytes']}")
        print(f"    Current Parent: {file_info['current_parent']}")
    if len(files_to_move) > 3:
        print("    ...")
else:
    print("  No files met the criteria.")

# --- Find Destination Folder ID ---
dest_folder_query = f"name='{destination_folder_name}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
dest_folder_response = gdrive.list_user_files(q=dest_folder_query)
destination_folder_id = dest_folder_response['files'][0]['id']


# --- Find Log Sheet ID ---
sheet_query = f"name='{log_sheet_name}' and mimeType='application/vnd.google-apps.spreadsheet' and trashed=false"
sheet_response = gdrive.list_user_files(q=sheet_query)
log_sheet_id = sheet_response['files'][0]['id']
if not log_sheet_id:
    raise ValueError(f"Could not get ID for Google Sheet '{log_sheet_name}'.")
print(f"Log Sheet: '{log_sheet_name}' (ID: {log_sheet_id})")

# Store necessary info for Block 2 (simulated - in real execution this data passes)


if not files_to_move:
    print('No entries are to be appended to the "Large Assets Log" sheet.')
else:
    # --- Process Each File ---
    for file_info in files_to_move:
        file_id = file_info['id']
        file_name = file_info['name']
        size_bytes = file_info['size_bytes']
        current_parent_id = file_info['current_parent']

        # 1. Move the file using files.update
        print(f"Moving file '{file_name}' (ID: {file_id}) from parent {current_parent_id} to {destination_folder_id}...")
        update_response = gdrive.update_file_metadata_or_content(
            fileId=file_id,
            addParents=destination_folder_id,
            removeParents=current_parent_id,
        )
        print(f"  Move API call for {file_id} completed. Response: {update_response}")

        # 2. Calculate file size in MB (rounded)
        size_mb = round(size_bytes / (1024 * 1024), 2)

        # 3. Append file info to the Google Sheet
        values_to_append = [[file_name, size_mb]]
        print(f"Appending to sheet '{log_sheet_id}', range '{target_range}': {values_to_append}")
        append_response = google_sheets.append_spreadsheet_values(
            spreadsheet_id=log_sheet_id,
            range=target_range,
            valueInputOption='USER_ENTERED',
            insertDataOption='INSERT_ROWS',
            values=values_to_append
        )
        print(f"  Append API call for {file_name} completed. Response: {append_response}")


Found 1 files to move and log:
  - File 1:
    ID: file_4
    Name: design_mockup_6MB.png
    Size (Bytes): 6291456
    Current Parent: file_3
Log Sheet: 'Large Assets Log' (ID: file_6)
Moving file 'design_mockup_6MB.png' (ID: file_4) from parent file_3 to file_5...
  Move API call for file_4 completed. Response: {'kind': 'drive#file', 'id': 'file_4', 'driveId': '', 'name': 'design_mockup_6MB.png', 'mimeType': 'application/octet-stream', 'parents': ['file_5'], 'createdTime': '2025-03-14T00:00:00Z', 'modifiedTime': '2025-03-14T00:00:00Z', 'trashed': False, 'starred': False, 'owners': ['john.doe@gmail.com'], 'size': '6291456', 'md5Checksum': '', 'sha1Checksum': '', 'sha256Checksum': '', 'imageMediaMetadata': {}, 'videoMediaMetadata': {}, 'permissions': [{'id': 'permission_file_4', 'role': 'owner', 'type': 'user', 'emailAddress': 'john.doe@gmail.com'}], 'enforceSingleParent': False, 'ignoreDefaultVisibility': False, 'keepRevisionForever': False, 'ocrLanguage': '', 'supportsAllDrives': Fa

# Final Assertion
1. Assert that all files in the "Campaign Assets – To Review" folder with "design" in their name (case-insensitive) and a file size greater than 5 MB have been moved to the "Campaign Assets – Heavy" folder.  
2. Assert that each of these moved files has its name and rounded file size (in MB) recorded as a new row in the "Large Assets Log" Google Sheet.  
3. Assert that none of the above files remain in the "Campaign Assets – To Review" folder.

In [7]:
from Scripts.assertions_utils import *
import gdrive
import google_sheets
from decimal import Decimal, ROUND_HALF_UP

# --- Config ---
folder_to_review_name = "Campaign Assets – To Review"
folder_heavy_name = "Campaign Assets – Heavy"
sheet_name = "Large Assets Log"
file_name_contains = "design"
min_file_size_bytes = 5 * 1024 * 1024  # 5 MB in bytes

def mb_rounded_half_up(num_bytes: int) -> int:
    mb = Decimal(num_bytes) / Decimal(1024 * 1024)
    return int(mb.quantize(Decimal("1"), rounding=ROUND_HALF_UP))

# --- 1) Find folders and sheet ---
folders = gdrive.list_user_files(q="mimeType='application/vnd.google-apps.folder' and trashed=false").get("files", [])
to_review = next((f for f in folders if compare_strings(f["name"], folder_to_review_name)), None)
heavy     = next((f for f in folders if compare_strings(f["name"], folder_heavy_name)), None)

sheet_files = gdrive.list_user_files(
    q=f"name='{sheet_name}' and mimeType='application/vnd.google-apps.spreadsheet' and trashed=false"
).get("files", [])
log_sheet = next((s for s in sheet_files if compare_strings(s["name"], sheet_name)), None)
spreadsheet_id = log_sheet["id"]

# Load sheet values with google_sheets API; pick first tab dynamically, default to first sheet if metadata not available.
try:
    meta = google_sheets.get_spreadsheet(spreadsheet_id=spreadsheet_id, includeGridData=True)
    data_dict = meta.get("data", {})
    a1_range = next(iter(data_dict.keys()), None)
    sheets = (meta or {}).get("sheets", [])
    values_resp = google_sheets.get_spreadsheet_values(spreadsheet_id=spreadsheet_id, range=a1_range) or {}
    sheet_values = values_resp.get("values", [])
    sheet_rows = sheet_values[1:] if len(sheet_values) > 1 else []  # drop header row
except Exception:
    a1_range = "A:B"
    sheet_rows = []

def is_logged(name: str, size_mb: int) -> bool:
    for row in sheet_rows:
        if len(row) < 2:
            continue
        r_name = (row[0] or "").strip()
        try:
            r_size_mb = int(Decimal(str(row[1])).quantize(Decimal("1"), rounding=ROUND_HALF_UP))
        except Exception:
            try:
                r_size_mb = int(Decimal(float(row[1])).quantize(Decimal("1"), rounding=ROUND_HALF_UP))
            except Exception:
                continue
        if compare_strings(r_name, name) and r_size_mb == size_mb:
            return True
    return False

# --- 2) Scope ONLY to the source/destination folders ---

# Files currently in "To Review" (should be none after the action)
review_q = (
    f"'{to_review['id']}' in parents and size > {min_file_size_bytes} and trashed=false "
    f"and mimeType!='application/vnd.google-apps.folder'"
)
review_raw = gdrive.list_user_files(q=review_q).get("files", [])
eligible_in_review_now = [f for f in review_raw if compare_is_string_subset(file_name_contains, f.get("name", ""))]

# Files currently in "Heavy" (should include all eligible files after the action)
heavy_q = (
    f"'{heavy['id']}' in parents and size > {min_file_size_bytes} and trashed=false "
    f"and mimeType!='application/vnd.google-apps.folder'"
)
heavy_raw = gdrive.list_user_files(q=heavy_q).get("files", [])
eligible_in_heavy_now = [f for f in heavy_raw if compare_is_string_subset(file_name_contains, f.get("name", ""))]

# (A) None of the eligible files should remain in "To Review"
assert not eligible_in_review_now, (
    "❌ Some eligible 'design' files (>5MB) still remain in 'Campaign Assets – To Review'. "
    f"Offenders: {[f.get('name') for f in eligible_in_review_now]}"
)

# (B) For each eligible file now in Heavy, ensure it’s logged in the sheet (name + rounded MB)
missing_in_log = []
for f in eligible_in_heavy_now:
    name = f.get("name", "")
    try:
        size_bytes = int(f.get("size", "0"))
    except Exception:
        size_bytes = 0
    size_mb = mb_rounded_half_up(size_bytes)
    if not is_logged(name, size_mb):
        missing_in_log.append(f"{name} ({size_mb} MB)")

assert not missing_in_log, (
    f"❌ Some moved files are not recorded in '{sheet_name}'. Missing: {missing_in_log}"
)

# (C) Sanity: ensure at least one eligible file ended up in Heavy (per base case expectation)
assert len(eligible_in_heavy_now) >= 1, "❌ Expected at least one eligible file in 'Campaign Assets – Heavy' but found none."
